In [0]:
import json
from pathlib import Path
import pandas as pd

In [0]:

dbutils.widgets.text("raw_root", "/Volumes/workspace/raw/tvmze", "Raw Root")
# Léalos en Python
raw_root = Path(dbutils.widgets.get("raw_root"))

In [0]:
%sql
DROP DATABASE IF EXISTS workspace.bronze CASCADE; 

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.bronze
COMMENT 'Capa Bronze: datos crudos procesados'

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.bronze.predios_registro")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.predios_registro (
    pk STRING,
    matricula STRING,
    fecha_radica_texto STRING,
    fecha_apertura_texto STRING,
    year_radica STRING,
    orip STRING,
    divipola STRING,
    departamento STRING,
    municipio STRING,
    tipo_predio_zona STRING,
    categoria_ruralidad_2024 STRING,
    num_anotacion STRING,
    dinamica_2024 STRING,
    cod_natujur STRING,
    nombre_natujur STRING,
    documento_justificativo STRING,
    count_a STRING,
    count_de STRING,
    predios_nuevos STRING,
    tiene_valor STRING,
    tiene_mas_de_un_valor STRING
)
USING DELTA
""")

In [0]:
candidate_files = sorted(raw_root.glob("**/tvmaze.json"))

records: list[dict] = []
for json_file in candidate_files:
    payload = json.loads(json_file.read_text(encoding="utf-8"))
    records.extend(payload)  # asumimos que payload siempre es lista

df = pd.json_normalize(records) if records else pd.DataFrame()
df_spark = spark.createDataFrame(df)


In [0]:
# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.bronze.predios_registro")